# IVF 임신 성공 예측 - CatBoost + SHAP 리팩토링 노트북

원본 `EDA.ipynb`의 흐름을 한 파일 안에서 재현 가능하게 정리한 버전입니다.

핵심 흐름:
1. 데이터 로드
2. 기본 전처리
3. IVF 도메인 기반 Feature Engineering
4. CatBoost 학습 및 평가
5. Threshold tuning
6. RandomizedSearchCV
7. SHAP summary / dependence plot
8. 최종 test inference / submission 생성

> 실험 중간 셀을 줄이고, 반복되던 전처리/학습/평가 코드를 함수화했습니다.


## 0. Import & Config


In [16]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import koreanize_matplotlib
except ImportError:
    pass

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

from catboost import CatBoostClassifier
import shap

RANDOM_STATE = 42
TARGET = "임신 성공 여부"
ID_COL = "ID"
TEST_SIZE = 0.2

TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
SUBMISSION_PATH = "sample_submission.csv"

NA_VALUES = ["None", "none", "NULL", "null"]


## 1. 데이터 로드


In [17]:
train_data = pd.read_csv(TRAIN_PATH, na_values=NA_VALUES)
test_data = pd.read_csv(TEST_PATH, na_values=NA_VALUES)
submission_data = pd.read_csv(SUBMISSION_PATH)

print("train:", train_data.shape)
print("test :", test_data.shape)
print("submission:", submission_data.shape)

display(train_data.head())


train: (256351, 69)
test : (90067, 68)
submission: (90067, 2)


,ID,시술 시기 코드,시술 당시 나이,임신 시도 또는 마지막 임신 경과 연수,시술 유형,특정 시술 유형,배란 자극 여부,배란 유도 유형,단일 배아 이식 여부,착상 전 유전 검사 사용 여부,...,기증 배아 사용 여부,대리모 여부,PGD 시술 여부,PGS 시술 여부,난자 채취 경과일,난자 해동 경과일,난자 혼합 경과일,배아 이식 경과일,배아 해동 경과일,임신 성공 여부
0,TRAIN_000000,TRZKPL,만18-34세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,3.0,NaN,0
1,TRAIN_000001,TRYBLT,만45-50세,NaN,IVF,ICSI,0,알 수 없음,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0
2,TRAIN_000002,TRVNRY,만18-34세,NaN,IVF,IVF,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,2.0,NaN,0
3,TRAIN_000003,TRJXFG,만35-37세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0
4,TRAIN_000004,TRVNRY,만18-34세,NaN,IVF,ICSI,1,기록되지 않은 시행,0.0,NaN,...,0.0,0.0,NaN,NaN,0.0,NaN,0.0,3.0,NaN,0


## 2. 기본 확인


In [18]:
display(train_data[TARGET].value_counts())
display(train_data[TARGET].value_counts(normalize=True))

missing_summary = train_data.isnull().sum().sort_values(ascending=False)
display(missing_summary[missing_summary > 0].head(30))


임신 성공 여부
0    190123
1     66228
Name: count, dtype: int64

임신 성공 여부
0    0.741651
1    0.258349
Name: proportion, dtype: float64

난자 해동 경과일                254915
PGS 시술 여부                254422
PGD 시술 여부                254172
착상 전 유전 검사 사용 여부         253633
임신 시도 또는 마지막 임신 경과 연수    246981
배아 해동 경과일                215982
난자 채취 경과일                 57488
난자 혼합 경과일                 53735
배아 이식 경과일                 43566
미세주입 후 저장된 배아 수            6291
해동된 배아 수                   6291
저장된 배아 수                   6291
미세주입 배아 이식 수               6291
대리모 여부                     6291
단일 배아 이식 여부                6291
착상 전 유전 진단 사용 여부           6291
배아 생성 주요 이유                6291
미세주입에서 생성된 배아 수            6291
이식된 배아 수                   6291
총 생성 배아 수                  6291
기증 배아 사용 여부                6291
미세주입된 난자 수                 6291
혼합된 난자 수                   6291
해동 난자 수                    6291
수집된 신선 난자 수                6291
파트너 정자와 혼합된 난자 수           6291
기증자 정자와 혼합된 난자 수           6291
신선 배아 사용 여부                6291
저장된 신선 난자 수                6291
동결 배아 사용 여부                6291
dtype: int64

## 3. 공통 전처리 / Feature Engineering 함수

원본 노트북에서 여러 셀에 흩어져 있던 작업을 `prepare_features()` 하나로 묶었습니다.

주의점:
- train/test 모두 같은 함수로 처리합니다.
- target leakage 방지를 위해 `TARGET`, `ID`는 feature에서 제외합니다.
- 컬럼이 없을 경우에도 에러가 나지 않게 안전 처리했습니다.


In [26]:
def make_ivf_features(df):
    df = df.copy()

    # -----------------------------
    # 1. time performed flag
    # -----------------------------
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일',
    ]

    for col in time_cols:
        if col in df.columns:
            df[f'{col}_performed'] = df[col].notnull().astype(int)

    df[time_cols] = df[time_cols].fillna(0)

    # -----------------------------
    # 2. count ordinal mapping
    # -----------------------------
    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        if col in df.columns:
            df[col] = df[col].map(count_map).astype(float)

    # -----------------------------
    # 3. safe division helper
    # -----------------------------
    def safe_divide(num, den):
        return np.where(den == 0, 0, num / den)

    # -----------------------------
    # 4. history ratio features
    # -----------------------------
    df['총_임신_성공률'] = safe_divide(df['총 임신 횟수'], df['총 시술 횟수'])
    df['IVF_임신_성공률'] = safe_divide(df['IVF 임신 횟수'], df['IVF 시술 횟수'])
    df['DI_임신_성공률'] = safe_divide(df['DI 임신 횟수'], df['DI 시술 횟수'])

    df['총_출산_전환률'] = safe_divide(df['총 출산 횟수'], df['총 임신 횟수'])
    df['IVF_출산_전환률'] = safe_divide(df['IVF 출산 횟수'], df['IVF 임신 횟수'])
    df['DI_출산_전환률'] = safe_divide(df['DI 출산 횟수'], df['DI 임신 횟수'])

    df['총_실패_횟수'] = df['총 시술 횟수'] - df['총 임신 횟수']
    df['IVF_실패_횟수'] = df['IVF 시술 횟수'] - df['IVF 임신 횟수']
    df['DI_실패_횟수'] = df['DI 시술 횟수'] - df['DI 임신 횟수']

    df['IVF_비중'] = safe_divide(df['IVF 시술 횟수'], df['총 시술 횟수'])
    df['DI_비중'] = safe_divide(df['DI 시술 횟수'], df['총 시술 횟수'])

    # -----------------------------
    # 5. embryo / egg utilization features
    # -----------------------------
    df['배아_이식률'] = safe_divide(df['이식된 배아 수'], df['총 생성 배아 수'])
    df['배아_저장률'] = safe_divide(df['저장된 배아 수'], df['총 생성 배아 수'])
    df['난자_배아_효율'] = safe_divide(df['총 생성 배아 수'], df['수집된 신선 난자 수'])

    df['총_배아_활용량'] = df['이식된 배아 수'] + df['저장된 배아 수']

    df['배아이식_수행여부'] = (df['이식된 배아 수'] > 0).astype(int)

    # -----------------------------
    # 6. age interaction features
    # -----------------------------
    high_age_groups = [
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]

    df['고령여부'] = df['시술 당시 나이'].isin(high_age_groups).astype(int)

    df['고령_이식배아_interaction'] = df['고령여부'] * df['이식된 배아 수']
    df['고령_생성배아_interaction'] = df['고령여부'] * df['총 생성 배아 수']
    df['고령_저장배아_interaction'] = df['고령여부'] * df['저장된 배아 수']
    df['고령_난자수_interaction'] = df['고령여부'] * df['수집된 신선 난자 수']
    df['고령_배아이식률_interaction'] = df['고령여부'] * df['배아_이식률']

    # -----------------------------
    # 7. final missing value handling
    # -----------------------------
    df = df.fillna(0)

    return df

In [41]:
def set_ivf_categories(X):
    X = X.copy()

    # 1. object/string 컬럼 먼저 문자열 통일
    obj_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()

    for col in obj_cols:
        X[col] = X[col].fillna("Missing").astype(str)

    # 2. binary columns 찾기
    binary_cols = []

    for col in X.columns:
        unique_values = set(X[col].dropna().unique())

        if unique_values == {0, 1}:
            binary_cols.append(col)

    # 3. 수동 categorical
    to_categorical_columns = [
        "불임 원인 - 여성 요인",
        "착상 전 유전 검사 사용 여부",
        "PGD 시술 여부",
        "PGS 시술 여부",
    ]

    to_categorical_columns = [
        col for col in to_categorical_columns
        if col in X.columns
    ]

    # 4. 최종 categorical 후보
    categorical_cols = list(set(obj_cols + binary_cols + to_categorical_columns))

    # 핵심: category 변환 전에 무조건 str 통일
    for col in categorical_cols:
        X[col] = X[col].fillna("Missing").astype(str).astype("category")

    return X

In [42]:
def prepare_features(df, is_train=True):
    df = df.copy()

    # ID 저장
    id_col = 'ID'

    if id_col in df.columns:
        ids = df[id_col]
        df = df.drop(columns=[id_col])
    else:
        ids = None

    # target 분리
    target_col = '임신 성공 여부'

    if is_train:
        y = df[target_col]
        X = df.drop(columns=[target_col])
    else:
        y = None
        X = df

    # feature engineering
    X = make_ivf_features(X)

    # categorical setting
    X = set_ivf_categories(X)

    return X, y, ids

## 4. Feature 생성


In [43]:
X, y, train_ids = prepare_features(train_data, is_train=True)
X_test_final, _, test_ids = prepare_features(test_data, is_train=False)

print("X:", X.shape)
print("y:", y.shape)
print("X_test_final:", X_test_final.shape)

# train/test 컬럼 정렬: train 기준으로 맞추고, test에 없는 컬럼은 0으로 채움
X_test_final = X_test_final.reindex(columns=X.columns, fill_value=0)

print("numeric columns:", X.select_dtypes(include=np.number).shape[1])
print("categorical columns:", X.select_dtypes(include="category").shape[1])
display(X.head())


X: (256351, 94)
y: (256351,)
X_test_final: (90067, 94)
numeric columns: 48
categorical columns: 46


,시술 시기 코드,시술 당시 나이,임신 시도 또는 마지막 임신 경과 연수,시술 유형,특정 시술 유형,배란 자극 여부,배란 유도 유형,단일 배아 이식 여부,착상 전 유전 검사 사용 여부,착상 전 유전 진단 사용 여부,...,배아_저장률,난자_배아_효율,총_배아_활용량,배아이식_수행여부,고령여부,고령_이식배아_interaction,고령_생성배아_interaction,고령_저장배아_interaction,고령_난자수_interaction,고령_배아이식률_interaction
0,TRZKPL,만18-34세,0.0,IVF,ICSI,1,기록되지 않은 시행,0.0,0.0,0.0,...,0.5,0.571429,4.0,1,0,0.0,0.0,0.0,0.0,0.0
1,TRYBLT,만45-50세,0.0,IVF,ICSI,0,알 수 없음,0.0,0.0,0.0,...,0.0,0.000000,0.0,0,1,0.0,0.0,0.0,1.0,0.0
2,TRVNRY,만18-34세,0.0,IVF,IVF,1,기록되지 않은 시행,0.0,0.0,0.0,...,0.0,0.625000,2.0,1,0,0.0,0.0,0.0,0.0,0.0
3,TRJXFG,만35-37세,0.0,IVF,ICSI,1,기록되지 않은 시행,0.0,0.0,0.0,...,0.0,0.000000,0.0,0,0,0.0,0.0,0.0,0.0,0.0
4,TRVNRY,만18-34세,0.0,IVF,ICSI,1,기록되지 않은 시행,0.0,0.0,0.0,...,0.0,0.857143,2.0,1,0,0.0,0.0,0.0,0.0,0.0


## 5. Train / Valid Split


In [44]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(X_train.shape, X_valid.shape)
print(y_train.value_counts(normalize=True))
print(y_valid.value_counts(normalize=True))


(205080, 94) (51271, 94)
임신 성공 여부
0    0.741652
1    0.258348
Name: proportion, dtype: float64
임신 성공 여부
0    0.741647
1    0.258353
Name: proportion, dtype: float64


In [37]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 256351 entries, 0 to 256350
Data columns (total 94 columns):
 #   Column                           Non-Null Count   Dtype   
---  ------                           --------------   -----   
 0   시술 시기 코드                         256351 non-null  category
 1   시술 당시 나이                         256351 non-null  category
 2   임신 시도 또는 마지막 임신 경과 연수            256351 non-null  float64 
 3   시술 유형                            256351 non-null  category
 4   특정 시술 유형                         256351 non-null  category
 5   배란 자극 여부                         256351 non-null  category
 6   배란 유도 유형                         256351 non-null  category
 7   단일 배아 이식 여부                      256351 non-null  category
 8   착상 전 유전 검사 사용 여부                 256351 non-null  category
 9   착상 전 유전 진단 사용 여부                 256351 non-null  category
 10  남성 주 불임 원인                       256351 non-null  category
 11  남성 부 불임 원인                       256351 non-null  category
 12 

## 6. Preprocessor 생성


In [45]:
def build_preprocessor(X):
    numeric_features = X.select_dtypes(
        include=np.number
        ).columns.tolist()

    categorical_features = X.select_dtypes(
        include=["category", "object"]
        ).columns.tolist()

    X[categorical_features] = (
        X[categorical_features]
        .astype(str)
        )
    numeric_transformer = Pipeline([
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])

    return preprocessor, numeric_features, categorical_features


preprocessor, numeric_features, categorical_features = build_preprocessor(X)

print("numeric:", len(numeric_features))
print("categorical:", len(categorical_features))


numeric: 48
categorical: 46


## 7. 평가 함수


In [46]:
def evaluate_proba(y_true, proba, threshold=0.5, name="model"):
    pred = (proba >= threshold).astype(int)

    metrics = {
        "model": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred),
        "precision": precision_score(y_true, pred),
        "recall": recall_score(y_true, pred),
        "roc_auc": roc_auc_score(y_true, proba),
    }

    print(f"[{name}] threshold={threshold}")
    for k, v in metrics.items():
        if k != "model":
            print(f"{k}: {v}")

    print("Confusion Matrix")
    print(confusion_matrix(y_true, pred))
    print("Classification Report")
    print(classification_report(y_true, pred))

    return metrics, pred


def find_best_threshold(y_true, proba, start=0.20, end=0.80, step=0.01):
    results = []
    for threshold in np.arange(start, end + step, step):
        pred = (proba >= threshold).astype(int)
        results.append({
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, pred),
            "f1": f1_score(y_true, pred),
            "precision": precision_score(y_true, pred),
            "recall": recall_score(y_true, pred),
        })

    threshold_df = pd.DataFrame(results).sort_values("f1", ascending=False)
    best_row = threshold_df.iloc[0]

    return best_row["threshold"], best_row["f1"], threshold_df


## 8. Baseline 모델 비교

빠른 기준선 확인용입니다. 오래 걸리면 이 섹션은 건너뛰어도 됩니다.


In [47]:
baseline_models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "decision_tree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    "gaussian_nb": GaussianNB(),
}

baseline_results = []

for name, model in baseline_models.items():
    pipe = Pipeline([
        ("preprocessor", build_preprocessor(X)[0]),
        ("model", model),
    ])

    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_valid)[:, 1]
    metrics, _ = evaluate_proba(y_valid, proba, name=name)
    baseline_results.append(metrics)

baseline_df = pd.DataFrame(baseline_results).sort_values("f1", ascending=False)
display(baseline_df)


[logistic_regression] threshold=0.5
threshold: 0.5
accuracy: 0.6169959626299467
f1: 0.509724614885277
precision: 0.3807960607304062
recall: 0.7706477427147819
roc_auc: 0.7291792483369621
Confusion Matrix
[[21426 16599]
 [ 3038 10208]]
Classification Report
              precision    recall  f1-score   support

           0       0.88      0.56      0.69     38025
           1       0.38      0.77      0.51     13246

    accuracy                           0.62     51271
   macro avg       0.63      0.67      0.60     51271
weighted avg       0.75      0.62      0.64     51271

[decision_tree] threshold=0.5
threshold: 0.5
accuracy: 0.6648007645647637
f1: 0.35468609191949535
precision: 0.3528313163006126
recall: 0.35656047108561073
roc_auc: 0.5643683354770591
Confusion Matrix
[[29362  8663]
 [ 8523  4723]]
Classification Report
              precision    recall  f1-score   support

           0       0.78      0.77      0.77     38025
           1       0.35      0.36      0.35     13246

,model,threshold,accuracy,f1,precision,recall,roc_auc
0,logistic_regression,0.5,0.616996,0.509725,0.380796,0.770648,0.729179
2,gaussian_nb,0.5,0.424587,0.466973,0.306945,0.975615,0.605901
1,decision_tree,0.5,0.664801,0.354686,0.352831,0.356560,0.564368


## 9. CatBoost 기본 학습

원본 노트북의 핵심 모델입니다. CatBoost에 `eval_set`을 깔끔하게 넣기 위해 여기서는 Pipeline 대신 전처리 결과 배열로 학습합니다.


In [48]:
preprocessor, numeric_features, categorical_features = build_preprocessor(X)

X_train_trans = preprocessor.fit_transform(X_train)
X_valid_trans = preprocessor.transform(X_valid)
X_test_trans_final = preprocessor.transform(X_test_final)

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="F1",
    random_seed=RANDOM_STATE,
    verbose=100,
    auto_class_weights="Balanced",
)

cat_model.fit(
    X_train_trans,
    y_train,
    eval_set=(X_valid_trans, y_valid),
    early_stopping_rounds=50,
)

cat_proba = cat_model.predict_proba(X_valid_trans)[:, 1]
cat_metrics, cat_pred = evaluate_proba(y_valid, cat_proba, threshold=0.5, name="CatBoost 기본")


TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

## 10. Threshold Tuning


In [ ]:
best_threshold, best_f1, threshold_df = find_best_threshold(y_valid, cat_proba)

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)

display(threshold_df.head(10))

cat_metrics_tuned, cat_pred_tuned = evaluate_proba(
    y_valid,
    cat_proba,
    threshold=best_threshold,
    name="CatBoost threshold tuned",
)


## 11. Feature Importance


In [ ]:
feature_names = preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": cat_model.feature_importances_,
}).sort_values("importance", ascending=False)

display(importance_df.head(30))


## 12. RandomizedSearchCV

원본 결과:

```text
Best Score: 0.517059203787733
Best Params: {'model__learning_rate': 0.05, 'model__l2_leaf_reg': 3, 'model__iterations': 300, 'model__depth': 7}
```

튜닝 개선폭이 작았기 때문에 기본값은 실행하지 않습니다.


In [ ]:
RUN_RANDOM_SEARCH = False

cat_pipe = Pipeline([
    ("preprocessor", build_preprocessor(X)[0]),
    ("model", CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="F1",
        random_seed=RANDOM_STATE,
        verbose=0,
        auto_class_weights="Balanced",
    )),
])

if RUN_RANDOM_SEARCH:
    param_dist = {
        "model__depth": [4, 5, 6, 7],
        "model__learning_rate": [0.03, 0.05, 0.07],
        "model__l2_leaf_reg": [3, 5, 7, 9],
        "model__iterations": [300, 500],
    }

    search = RandomizedSearchCV(
        estimator=cat_pipe,
        param_distributions=param_dist,
        n_iter=10,
        scoring="f1",
        cv=3,
        n_jobs=-1,
        verbose=2,
        random_state=RANDOM_STATE,
    )

    search.fit(X, y)

    print("Best Score:", search.best_score_)
    print("Best Params:", search.best_params_)

    search_result_df = pd.DataFrame(search.cv_results_)[
        [
            "mean_test_score",
            "std_test_score",
            "param_model__depth",
            "param_model__learning_rate",
            "param_model__l2_leaf_reg",
            "param_model__iterations",
        ]
    ].sort_values("mean_test_score", ascending=False)

    display(search_result_df.head(10))
else:
    print("RandomizedSearchCV는 시간이 걸리므로 기본값은 실행하지 않습니다. RUN_RANDOM_SEARCH=True로 바꾸면 실행됩니다.")


## 13. Best Params 기반 CatBoost 재학습


In [ ]:
best_cat_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=7,
    l2_leaf_reg=3,
    loss_function="Logloss",
    eval_metric="F1",
    random_seed=RANDOM_STATE,
    verbose=100,
    auto_class_weights="Balanced",
)

best_cat_model.fit(
    X_train_trans,
    y_train,
    eval_set=(X_valid_trans, y_valid),
    early_stopping_rounds=50,
)

best_proba = best_cat_model.predict_proba(X_valid_trans)[:, 1]
best_metrics, best_pred = evaluate_proba(y_valid, best_proba, threshold=0.5, name="CatBoost best params")

best_t, best_t_f1, best_threshold_df = find_best_threshold(y_valid, best_proba)
print("Best threshold:", best_t)
print("Best F1:", best_t_f1)


## 14. SHAP Summary Plot

주의:
- `shap_values`를 만든 데이터와 plot에 넣는 데이터의 row 수가 반드시 같아야 합니다.
- 여기서는 `X_valid_trans` 기준으로 통일합니다.


In [ ]:
X_valid_shap = X_valid_trans
if hasattr(X_valid_shap, "toarray"):
    X_valid_shap = X_valid_shap.toarray()

explainer = shap.TreeExplainer(best_cat_model)
shap_values = explainer.shap_values(X_valid_shap)

shap.summary_plot(
    shap_values,
    X_valid_shap,
    feature_names=feature_names,
)


## 15. SHAP Bar Plot


In [ ]:
shap.summary_plot(
    shap_values,
    X_valid_shap,
    feature_names=feature_names,
    plot_type="bar",
)


## 16. SHAP Dependence Plot

threshold/bucket feature를 찾기 위한 핵심 구간입니다.


In [ ]:
def search_feature_names(keyword, feature_names=feature_names):
    return [f for f in feature_names if keyword in f]

search_feature_names("배아")[:50]


In [ ]:
important_features = [
    "num__배아_이식_집중도",
    "num__이식된 배아 수",
    "num__배아이식_수행여부",
    "num__고령_난자수_interaction",
    "num__배아 이식 경과일",
]

for feature in important_features:
    if feature in feature_names:
        shap.dependence_plot(
            feature,
            shap_values,
            X_valid_shap,
            feature_names=feature_names,
            interaction_index=None,
        )
    else:
        print(f"없는 feature라서 skip: {feature}")


## 17. 최종 모델 전체 학습

검증까지 끝난 뒤 제출용으로 전체 train 데이터에 다시 학습합니다.


In [ ]:
FINAL_THRESHOLD = float(best_t) if "best_t" in globals() else 0.5

final_preprocessor, _, _ = build_preprocessor(X)
X_all_trans = final_preprocessor.fit_transform(X)
X_test_final_trans = final_preprocessor.transform(X_test_final)

final_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=7,
    l2_leaf_reg=3,
    loss_function="Logloss",
    eval_metric="F1",
    random_seed=RANDOM_STATE,
    verbose=100,
    auto_class_weights="Balanced",
)

final_model.fit(X_all_trans, y)

print("Final model fitted.")
print("FINAL_THRESHOLD:", FINAL_THRESHOLD)


## 18. Test Inference / Submission 생성


In [ ]:
test_proba = final_model.predict_proba(X_test_final_trans)[:, 1]
test_pred = (test_proba >= FINAL_THRESHOLD).astype(int)

submission = submission_data.copy()

# 대회 제출 컬럼명이 다를 수 있으므로 마지막 컬럼에 예측값 입력
target_submit_col = submission.columns[-1]
submission[target_submit_col] = test_pred

submission.to_csv("submission_catboost_shap_refactored.csv", index=False)

print("saved: submission_catboost_shap_refactored.csv")
display(submission.head())


## 19. 다음 실험 후보

현재 튜닝/threshold에서 개선폭이 작았으므로 다음 병목은 feature engineering일 가능성이 큽니다.

추천 실험:
1. `배아_이식_집중도_bucket` 경계값을 SHAP dependence plot 기준으로 재조정
2. `이식된 배아 수 >= 2`, `>=3` bucket 비교
3. 고령군 × 배아 생성/이식/저장 계열 interaction 정리
4. PR AUC 추가 확인
5. StratifiedKFold OOF 기반으로 threshold 안정성 확인
